# WAF Security
- Assessment Audit

This notebook assesses the Security pillar using the **4 Key Topics** structure.

## Scoring Model (1-5 Scale)
| Score | Rating | Description |
|-------|--------|-------------|
| 5 | Excellent | Best practices fully implemented |
| 4 | Good | Most best practices, minor gaps |
| 3 | Moderate | Basic implementation |
| 2 | Limited | Minimal implementation |
| 1 | Poor | Not implemented |

## 4 Key Topics

1. **Secure**
- Authentication, Network Security, Encryption

2. **Govern**
- Access Control, Data Classification, Data Protection

3. **Comply**
- Audit Logging, Data Residency, Trust Center

4. **Monitor**
- Security Dashboards, Alerting, SIEM Integration

In [31]:
SCHEMA = "IE"
ACCOUNT_ID = 100058
DATABASE = "SNOWHOUSE_IMPORT"
print(f"Target: {DATABASE}.{SCHEMA} | Account ID: {ACCOUNT_ID}")

Target: SNOWHOUSE_IMPORT.IE | Account ID: 100058


In [32]:
from snowflake.snowpark import Session
import pandas as pd
import os

connection_name = os.getenv("SNOWFLAKE_CONNECTION_NAME") or "snowhouse"
if 'session' not in globals():
    session = Session.builder.config("connection_name", connection_name).create()
print(f"Connected via: {connection_name}")

Connected via: snowhouse


---
# Key Topic 1: SECURE

**Protect access to your Snowflake environment.**

| Sub-Topic | DDM | Source |
|-----------|-----|--------|
| Authentication | Yes | USER_ETL_V, INTEGRATION_ETL_V |
| Network Security | Yes | NETWORK_POLICY_ETL_V |
| Encryption | Manual | N/A |

In [33]:
# Authentication Assessment
auth_df = session.sql(f"""
SELECT COUNT(*) as total_users,
    COUNT(CASE WHEN MFA_AUTHENTICATORS_COUNT > 0 OR DEFAULT_MFA_TYPE_ID IS NOT NULL THEN 1 END) as mfa_users,
    COUNT(CASE WHEN TYPE = 3 THEN 1 END) as service_accounts,
    COUNT(CASE WHEN TYPE = 1 OR TYPE IS NULL THEN 1 END) as human_users
FROM {DATABASE}.{SCHEMA}.USER_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL
""").to_pandas()
print("=== Authentication ===")
print(f"Human Users: {auth_df['HUMAN_USERS'].iloc[0]} | With MFA: {auth_df['MFA_USERS'].iloc[0]}")
mfa_pct = (auth_df['MFA_USERS'].iloc[0] / auth_df['HUMAN_USERS'].iloc[0] * 100) if auth_df['HUMAN_USERS'].iloc[0] > 0 else 0
mfa_score = 5 if mfa_pct >= 90 else 4 if mfa_pct >= 70 else 3 if mfa_pct >= 50 else 2 if mfa_pct >= 20 else 1
print(f"MFA Adoption: {mfa_pct:.1f}% | Score: {mfa_score}/5")

=== Authentication ===
Human Users: 7541 | With MFA: 1
MFA Adoption: 0.0% | Score: 1/5


In [34]:
# SSO Integration (TYPE_ID 34=SCIM, 50=SAML2, 42=External OAuth for SSO)
sso_df = session.sql(f"""
SELECT COUNT(*) as sso_count
FROM {DATABASE}.{SCHEMA}.INTEGRATION_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL
  AND TYPE_ID IN (34, 50, 42)
""").to_pandas()
sso_count = sso_df['SSO_COUNT'].iloc[0]
print(f"=== SSO Integration ===")
print(f"Integrations: {sso_count}")
sso_score = 5 if sso_count >= 2 else 4 if sso_count >= 1 else 1
print(f"Score: {sso_score}/5")

=== SSO Integration ===
Integrations: 3
Score: 5/5


In [35]:
# Network Security
net_df = session.sql(f"""
SELECT COUNT(*) as policy_count
FROM {DATABASE}.{SCHEMA}.NETWORK_POLICY_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL
""").to_pandas()
net_count = net_df['POLICY_COUNT'].iloc[0]
print(f"=== Network Security ===\nNetwork Policies: {net_count}")
net_score = 5 if net_count >= 3 else 4 if net_count >= 2 else 3 if net_count >= 1 else 1
print(f"Score: {net_score}/5")

=== Network Security ===
Network Policies: 4
Score: 5/5


## Manual Assessment: Encryption

**Questions:**
1. Tri-Secret Secure needed?
2. BYOK required?
3. End-to-end encryption verified?

**Scoring:** 5=BYOK+documented | 3=Default understood | 1=No awareness

**Manual Score:** _____

In [36]:
encryption_score = 3  # Update from manual
auth_avg = round((mfa_score + sso_score) / 2, 1)
print("="*60 + "\nKEY TOPIC 1: SECURE - Summary\n" + "="*60)
print(f"  MFA: {mfa_score}/5 | SSO: {sso_score}/5 | Network: {net_score}/5 | Encryption: {encryption_score}/5")
secure_score = round((auth_avg + net_score + encryption_score) / 3, 1)
print(f"\n📊 SECURE TOPIC SCORE: {secure_score}/5")

KEY TOPIC 1: SECURE - Summary
  MFA: 1/5 | SSO: 5/5 | Network: 5/5 | Encryption: 3/5

📊 SECURE TOPIC SCORE: 3.7/5


---
# Key Topic 2: GOVERN

**Control who can access what data.**

| Sub-Topic | DDM | Source |
|-----------|-----|--------|
| Access Control | Yes | GRANT_REC_ETL_V |
| Data Classification | Yes | TAG_ETL_V |
| Data Protection | Yes | POLICY_ETL_V |

In [39]:
# Access Control
roles_df = session.sql(f"""
SELECT COUNT(DISTINCT GRANTEE_ROLE_ID) as total_roles
FROM {DATABASE}.{SCHEMA}.GRANT_REC_ETL_V
WHERE GRANTEE_ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL
""").to_pandas()
total_roles = roles_df['TOTAL_ROLES'].iloc[0]
print(f"=== Access Control ===\nRoles with grants: {total_roles}")
access_score = 5 if total_roles >= 20 else 4 if total_roles >= 10 else 3 if total_roles >= 5 else 2
print(f"Score: {access_score}/5")

=== Access Control ===
Roles with grants: 0
Score: 2/5


In [40]:
# Data Classification
tags_df = session.sql(f"""
SELECT COUNT(DISTINCT ID) as total_tags,
    COUNT(CASE WHEN UPPER(NAME) LIKE '%PII%' OR UPPER(NAME) LIKE '%SENSITIVE%' OR UPPER(NAME) LIKE '%CLASSIF%' THEN ID END) as class_tags
FROM {DATABASE}.{SCHEMA}.TAG_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL
""").to_pandas()
class_tags = tags_df['CLASS_TAGS'].iloc[0]
print(f"=== Data Classification ===\nClassification Tags: {class_tags}")
class_score = 5 if class_tags >= 3 else 4 if class_tags >= 2 else 3 if class_tags >= 1 else 2 if tags_df['TOTAL_TAGS'].iloc[0] >= 1 else 1
print(f"Score: {class_score}/5")

=== Data Classification ===
Classification Tags: 0
Score: 2/5


In [41]:
# Data Protection
policy_df = session.sql(f"""
SELECT COUNT(CASE WHEN KIND_ID = 1 THEN 1 END) as masking,
    COUNT(CASE WHEN KIND_ID = 2 THEN 1 END) as row_access
FROM {DATABASE}.{SCHEMA}.POLICY_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL
""").to_pandas()
total_policies = policy_df['MASKING'].iloc[0] + policy_df['ROW_ACCESS'].iloc[0]
print(f"=== Data Protection ===\nMasking: {policy_df['MASKING'].iloc[0]} | Row Access: {policy_df['ROW_ACCESS'].iloc[0]}")
prot_score = 5 if total_policies >= 5 else 4 if total_policies >= 3 else 3 if total_policies >= 1 else 1
print(f"Score: {prot_score}/5")

=== Data Protection ===
Masking: 126 | Row Access: 44
Score: 5/5


In [42]:
print("="*60 + "\nKEY TOPIC 2: GOVERN - Summary\n" + "="*60)
print(f"  Access Control: {access_score}/5 | Classification: {class_score}/5 | Protection: {prot_score}/5")
govern_score = round((access_score + class_score + prot_score) / 3, 1)
print(f"\n📊 GOVERN TOPIC SCORE: {govern_score}/5")

KEY TOPIC 2: GOVERN - Summary
  Access Control: 2/5 | Classification: 2/5 | Protection: 5/5

📊 GOVERN TOPIC SCORE: 3.0/5


---
# Key Topic 3: COMPLY

**Meet regulatory and audit requirements.**

| Sub-Topic | DDM | Manual |
|-----------|-----|--------|
| Audit Logging | Partial | Yes |
| Data Residency | No | Yes |
| Trust Center | No | Yes |

## Manual Assessment: Audit Logging

**Questions:**
1. Access History monitored?
2. Retention period?
3. Logs exported?

**Scoring:** 5=Monitored+exported | 3=Basic monitoring | 1=None

**Manual Score:** _____

## Manual Assessment: Data Residency

**Questions:**
1. Sovereignty requirements met?
2. Cross-region policies defined?
3. Approved regions only?

**Scoring:** 5=Documented+enforced | 3=Informal | 1=Not considered

**Manual Score:** _____

## Manual Assessment: Trust Center

**Questions:**
1. Trust Center reviewed?
2. Recommendations acted upon?
3. Compliance cadence?

**Scoring:** 5=Regular reviews | 3=Occasional | 1=Never

**Manual Score:** _____

In [43]:
audit_score = 3; residency_score = 3; trust_score = 3  # Update from manual
print("="*60 + "\nKEY TOPIC 3: COMPLY - Summary\n" + "="*60)
print(f"  Audit: {audit_score}/5 | Residency: {residency_score}/5 | Trust Center: {trust_score}/5")
comply_score = round((audit_score + residency_score + trust_score) / 3, 1)
print(f"\n📊 COMPLY TOPIC SCORE: {comply_score}/5")

KEY TOPIC 3: COMPLY - Summary
  Audit: 3/5 | Residency: 3/5 | Trust Center: 3/5

📊 COMPLY TOPIC SCORE: 3.0/5


---
# Key Topic 4: MONITOR

**Detect and respond to security events.**

| Sub-Topic | DDM | Manual |
|-----------|-----|--------|
| Security Dashboards | No | Yes |
| Alerting | Yes |
- |
| SIEM Integration | No | Yes |

## Manual Assessment: Security Dashboards

**Questions:**
1. Dashboards built?
2. Failed logins tracked?
3. Privilege usage monitored?

**Scoring:** 5=Comprehensive+reviewed | 3=Basic | 1=None

**Manual Score:** _____

In [44]:
# Alerting
alerts_df = session.sql(f"""
SELECT COUNT(*) as active_alerts
FROM {DATABASE}.{SCHEMA}.ALERT_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL
""").to_pandas()
active_alerts = alerts_df['ACTIVE_ALERTS'].iloc[0]
print(f"=== Alerting ===")
print(f"Active Alerts: {active_alerts}")
alerts_score = 5 if active_alerts >= 10 else 4 if active_alerts >= 5 else 3 if active_alerts >= 2 else 2 if active_alerts >= 1 else 1
print(f"Score: {alerts_score}/5")

=== Alerting ===
Active Alerts: 3
Score: 3/5


## Manual Assessment: SIEM Integration

**Questions:**
1. SIEM integration?
2. Events exported?
3. Centralized logging?

**Scoring:** 5=Full integration | 3=Partial export | 1=None

**Manual Score:** _____

In [45]:
dashboard_score = 3; siem_score = 3  # Update from manual
print("="*60 + "\nKEY TOPIC 4: MONITOR - Summary\n" + "="*60)
print(f"  Dashboards: {dashboard_score}/5 | Alerting: {alerts_score}/5 | SIEM: {siem_score}/5")
monitor_score = round((dashboard_score + alerts_score + siem_score) / 3, 1)
print(f"\n📊 MONITOR TOPIC SCORE: {monitor_score}/5")

KEY TOPIC 4: MONITOR - Summary
  Dashboards: 3/5 | Alerting: 3/5 | SIEM: 3/5

📊 MONITOR TOPIC SCORE: 3.0/5


---
# Overall Assessment Summary

In [46]:
print("="*70 + "\nSECURITY - OVERALL ASSESSMENT\n" + "="*70)
topics = {"1. Secure": secure_score, "2. Govern": govern_score, "3. Comply": comply_score, "4. Monitor": monitor_score}
for topic, score in topics.items():
    bar = "█" * int(score) + "░" * (5 - int(score))
    status = "✅" if score >= 4 else "⚠️" if score >= 3 else "❌"
    print(f"{status} {topic}: {bar} {score}/5")
overall_score = round(sum(topics.values()) / len(topics), 1)
print(f"\n🏆 OVERALL SECURITY SCORE: {overall_score}/5")

SECURITY - OVERALL ASSESSMENT
⚠️ 1. Secure: ███░░ 3.7/5
⚠️ 2. Govern: ███░░ 3.0/5
⚠️ 3. Comply: ███░░ 3.0/5
⚠️ 4. Monitor: ███░░ 3.0/5

🏆 OVERALL SECURITY SCORE: 3.2/5


In [47]:
prompt = f"""WAF Security advisor. Provide:
1. EXECUTIVE SUMMARY (2-3 sentences)
2. CRITICAL FINDINGS (3-5 issues with priority)
3. QUICK WINS (3-5 actions for 1 week)

Security: {overall_score}/5 | Topics: {topics}"""
result = session.sql(f"SELECT SNOWFLAKE.CORTEX.COMPLETE('claude-3-5-sonnet', '{prompt.replace(chr(39), chr(39)+chr(39))}') as response").collect()
print("🤖 AI INSIGHTS\n" + result[0]['RESPONSE'])

🤖 AI INSIGHTS
WAF SECURITY ADVISORY REPORT

1. EXECUTIVE SUMMARY
The Web Application Firewall (WAF) implementation shows moderate security maturity with a score of 3.2/5, indicating room for significant improvements. Critical gaps exist in rule customization, monitoring capabilities, and incident response procedures, exposing potential vulnerabilities to modern web attacks. Immediate attention is required to strengthen the WAF configuration and operational processes.

2. CRITICAL FINDINGS
Priority 1 (High): Outdated WAF rule sets not aligned with current threat landscape
- Default rules without custom tuning for application-specific threats
- Missing protection against emerging attack patterns

Priority 2 (High): Insufficient monitoring and alerting mechanisms
- Limited visibility into blocked requests and false positives
- Delayed response to security incidents

Priority 3 (Medium): Incomplete WAF change management process
- Uncontrolled rule modifications
- Lack of testing environmen

---
## Coverage Matrix

| Topic | Sub-Topic | DDM | Manual | Source |
|-------|-----------|-----|--------|--------|
| Secure | MFA | ✅ |
- | USER_ETL_V |
| Secure | SSO | ✅ |
- | INTEGRATION_ETL_V |
| Secure | Network | ✅ |
- | NETWORK_POLICY_ETL_V |
| Secure | Encryption |
- | ✅ | N/A |
| Govern | Access Control | ✅ |
- | GRANT_REC_ETL_V |
| Govern | Classification | ✅ |
- | TAG_ETL_V |
| Govern | Protection | ✅ |
- | POLICY_ETL_V |
| Comply | Audit |
- | ✅ | N/A |
| Comply | Residency |
- | ✅ | N/A |
| Comply | Trust Center |
- | ✅ | N/A |
| Monitor | Dashboards |
- | ✅ | N/A |
| Monitor | Alerting | ✅ |
- | ALERT_ETL_V |
| Monitor | SIEM |
- | ✅ | N/A |

**Summary:** 7 DDM | 6 Manual Only